# SageMaker Training Jobs — Parallel Generator Training

Use this notebook inside a **SageMaker Studio** or **SageMaker Notebook Instance** to:
1. Upload the raw dataset to S3
2. Launch three parallel SageMaker Training Jobs (one per generator)
3. Download and compare the resulting synthetic datasets

> **Local dev**: If you don't need SageMaker, use `notebooks/01_full_pipeline.ipynb` instead.

In [ ]:
import sagemaker
import boto3
import yaml
from pathlib import Path

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

print(f'SageMaker role: {role}')
print(f'Region: {region}')

## Step 1: Upload raw dataset to S3

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.s3_utils import provision_buckets, upload_file

# Create S3 buckets if they don't exist
provision_buckets(cfg)

raw_bucket = cfg['aws']['buckets']['raw']
prefix = cfg['aws']['prefix']

# Upload raw dataset
raw_uri = upload_file(
    local_path='../data/raw/creditcard.csv',
    bucket=raw_bucket,
    s3_key=f'{prefix}/raw/creditcard.csv',
    region=region,
)
print(f'Raw data uploaded: {raw_uri}')

## Step 2: Launch parallel training jobs

One training job per generator, launched in parallel.

In [ ]:
from sagemaker.sklearn.estimator import SKLearn

# We use the generic SageMaker Python SDK with a custom script
# In a real setup, package src/ as a SageMaker training script

INSTANCE_TYPE = 'ml.m5.2xlarge'  # 8 vCPU, 32 GB — good for tabular training

base_hyperparams = {
    'raw_s3_path': raw_uri,
    'target_col': cfg['data']['target_column'],
    'train_frac': str(cfg['splits']['train']),
    'val_frac': str(cfg['splits']['val']),
    'random_state': str(cfg['splits']['random_state']),
    'synthetic_bucket': cfg['aws']['buckets']['synthetic'],
    'prefix': prefix,
}

generators_config = [
    {
        'name': 'copula',
        'hyperparams': {**base_hyperparams, 'generator': 'copula'},
    },
    {
        'name': 'ctgan',
        'hyperparams': {
            **base_hyperparams,
            'generator': 'ctgan',
            'epochs': str(cfg['generators']['ctgan']['epochs']),
            'batch_size': str(cfg['generators']['ctgan']['batch_size']),
        },
    },
    {
        'name': 'tvae',
        'hyperparams': {
            **base_hyperparams,
            'generator': 'tvae',
            'epochs': str(cfg['generators']['tvae']['epochs']),
            'batch_size': str(cfg['generators']['tvae']['batch_size']),
        },
    },
]

# NOTE: In a full implementation, point entry_point to a SageMaker-compatible
# training script that wraps src/generators/train_generators.py
print('Training job configs prepared.')
print('To launch: create one SKLearn/PyTorch estimator per generator and call .fit() in parallel.')
print('See SageMaker docs: https://docs.aws.amazon.com/sagemaker/latest/dg/train-model.html')

## Step 3: Download synthetic datasets from S3

In [ ]:
from src.utils.s3_utils import download_file
import pandas as pd

synth_datasets = {}
for gen_name in ['copula', 'ctgan', 'tvae']:
    local_path = f'../data/synthetic/{gen_name}_synthetic.csv'
    try:
        download_file(
            bucket=cfg['aws']['buckets']['synthetic'],
            s3_key=f"{prefix}/synthetic/{gen_name}_synthetic.csv",
            local_path=local_path,
            region=region,
        )
        synth_datasets[gen_name] = pd.read_csv(local_path)
        print(f'  {gen_name}: {len(synth_datasets[gen_name]):,} rows')
    except Exception as e:
        print(f'  {gen_name}: not yet available ({e})')

print('\nNote: Run evaluation notebook (01_full_pipeline.ipynb) with these datasets.')